# Organizing budget data
These codes in this file aim to organize budget data.   
That's because processed data is recording all operations (e.g. add, repeal and amend) of trailer bills and has some duplication.

In [13]:
import pandas as pd

In [14]:
# Designate a target csv file
processed_data = "2023_budget_bill_final.csv"

In [15]:
# Convert csv files into dataframes
target_df = pd.read_csv(processed_data)
target_df.head()

,level,agency,item_number,program_code,object_code,fund_code,fund_name,schedule_group,name,amount_numeric,revised_amount,trailer_bill_source,action_type
0,program,LEGISLATIVE/JUDICIAL/EXECUTIVE,0110-001-0001,960.0,EMPTY,1.0,NaN,1.0,Support of the Senate,177325000.0,177325000,Original,Original
1,object,LEGISLATIVE/JUDICIAL/EXECUTIVE,0110-001-0001,960.0,101001.0,1.0,NaN,1.0,Salaries of Senators,-6751000.0,-6751000,Original,Original
2,object,LEGISLATIVE/JUDICIAL/EXECUTIVE,0110-001-0001,960.0,317295.0,1.0,NaN,1.0,Mileage,-11000.0,-11000,Original,Original
3,object,LEGISLATIVE/JUDICIAL/EXECUTIVE,0110-001-0001,960.0,317292.0,1.0,NaN,1.0,Expenses,-1712000.0,-1712000,Original,Original
4,object,LEGISLATIVE/JUDICIAL/EXECUTIVE,0110-001-0001,960.0,500004.0,1.0,NaN,1.0,Operating Expenses,-168851000.0,-168851000,Original,Original


In [16]:
# ---------------------------------------------------------
# Define the unique keys for budget line items
# ---------------------------------------------------------
unique_keys = ['item_number', 'program_code', 'object_code', 'fund_code']

In [17]:
# ---------------------------------------------------------
# ① Define the priority of the bills (higher number means newer)
# ---------------------------------------------------------
bill_priority = {'SB105': 3, 'SB104': 2, 'AB102': 1, 'Original': 0}
target_df['priority'] = target_df['trailer_bill_source'].map(bill_priority)

# Identify the maximum priority for each item_number
target_df['max_priority'] = target_df.groupby('item_number')['priority'].transform('max')

In [18]:
# ---------------------------------------------------------
# ② Extract only the latest rows for each item_number that are not marked as "Amended (Dropped/Old)"
# ---------------------------------------------------------
# Extract rows where the priority is the maximum for that item_number and the action_type is not "Amended (Dropped/Old)"
df_latest_only = target_df[
    (target_df['priority'] == target_df['max_priority']) & 
    (target_df['action_type'] != "Amended (Dropped/Old)")
].copy()

# ---------------------------------------------------------
# ③ Aggregate the amounts by unique keys, summing up the revised_amount (which includes reimbursements) and keeping reference information
# ---------------------------------------------------------
df_calculated = df_latest_only.groupby(unique_keys, as_index=False).agg({
    'revised_amount': 'sum',      
    'amount_numeric': 'first',    
    'agency': 'first',
    'fund_name': 'first',
    'trailer_bill_source': 'first'
})

# ---------------------------------------------------------
# ④ Organize the data for output, ensuring that subtotals without details are hidden
# ---------------------------------------------------------
# Hide subtotals that have no details (i.e., where program_code and object_code are "EMPTY" but there are no corresponding items with details)
items_with_details = set(
    df_calculated[
        (df_calculated["program_code"] != "EMPTY") | 
        (df_calculated["object_code"] != "EMPTY")
    ]["item_number"]
)

def filter_subtotals(row):
    if row["program_code"] == "EMPTY" and row["object_code"] == "EMPTY":
        return row["item_number"] not in items_with_details
    return True

df_final_csv = df_calculated[df_calculated.apply(filter_subtotals, axis=1)].copy()

# ---------------------------------------------------------
# ⑤ Output the final data to a new CSV file, sorted by unique keys
# ---------------------------------------------------------
cols_to_drop = ['amount_numeric', 'agency', 'fund_name', 'trailer_bill_source']
df_final_csv = df_final_csv.drop(columns=cols_to_drop)

df_final_csv = df_final_csv.sort_values(by=unique_keys)
df_final_csv.to_csv("2023_Budget_Final_Amounts.csv", index=False, encoding='utf-8-sig')

print(f"Complete: {len(df_final_csv)} rows saved.")

Complete: 2613 rows saved.
